
# Step 4. PRS Integration and Multimodal Feature Assembly

## 1. Overview

This notebook integrates polygenic risk scores (PRS), medical image activation vectors, and demographic and clinical characteristics into a participant-level dataset for subsequent type 2 diabetes (T2D) prediction.

The notebook does not train or evaluate a multimodal prediction model. Its primary objective is to prepare a consistent set of predictors for the subsequent XGBoost and TabPFN analyses.

## 2. PRS Integration

### Original Study

In the original study, PRS integration and parameter tuning were performed using an additional cohort of 7,421 participants with genetic data who were not included in the imaging cohort.

Logistic regression was used to combine ancestry-specific PRS, allowing the integration weights to be estimated separately from the imaging cohort used for multimodal prediction.

### GitHub Demonstration

The demonstration dataset contains only 100 participants and does not include the additional cohort used for PRS integration in the original study.

Therefore, the original logistic-regression-based integration procedure cannot be reproduced using the same data design.

For this demonstration, the ancestry-specific PRS-CSx scores are combined using predefined equal coefficients:

**PRS-CSx combined = PRS-CSx EAS + PRS-CSx EUR + PRS-CSx SAS**

This 1:1:1 combination is performed directly on the original ancestry-specific PRS scores, without estimating weights from T2D outcomes in the demonstration cohort.

Equal coefficients do not necessarily imply equal statistical contributions when the ancestry-specific scores have different distributions or scales.

### Alternative PRS Integration Strategies

Researchers with suitable datasets may use other PRS integration strategies, including logistic regression.

For example, logistic integration weights could be estimated using Training1 and Training2, with performance assessed using Validation. Testing must remain untouched during weight estimation and model selection.

When the resulting PRS is used as an input to downstream models trained on Training2, out-of-fold predictions or another appropriate cross-fitting strategy should be used to avoid target leakage from supervised PRS integration.

The Validation dataset should not be interpreted as an independent test set if it is reused for both PRS integration selection and downstream model selection.

**This notebook implements only the predefined 1:1:1 PRS-CSx combination. No supervised PRS integration is performed.**

## 3. Feature Sources

| Source | Features per participant |
|---|---:|
| ABD-US GNN activation vector | 1,024 |
| BMD left hip DenseNet121 activation vector | 1,024 |
| BMD right hip DenseNet121 activation vector | 1,024 |
| BMD spine DenseNet121 activation vector | 1,024 |
| Age, sex, family history | 3 |
| Combined PRS-CSx score | 1 |
| **Total** | **4,100** |

All activation vectors are generated in the preceding DenseNet121 and GNN notebooks.

The dataset also contains `Demo_ID`, `Dataset`, and `T2D` for participant identification, data partitioning, and outcome definition. These three columns are not predictors.

## 4. Data Partition and Leakage Prevention

The original demonstration partitions are preserved:

| Dataset | Participants | Subsequent use |
|---|---:|---|
| Training1 | 32 | DenseNet training |
| Training2 | 32 | GNN and multimodal model training |
| Validation | 16 | Model selection |
| Testing | 20 | Final demonstration evaluation |

No participant is reassigned to a different dataset.

The predefined PRS combination uses no T2D labels, so no PRS integration coefficients are fitted on Training1, Training2, Validation, or Testing.

The assembled dataset preserves the existing partitions for subsequent model development and evaluation.

## 5. Output

The notebook generates a participant-level CSV file containing:

- 100 participants.
- 4,100 predictors.
- Three additional columns: `Demo_ID`, `Dataset`, and `T2D`.

The output is saved to:

`Outputs/Step4_Feature_Assembly/Multimodal_100_samples_PRSCSx_EqualWeight.csv`

The original demonstration data and the preceding model outputs are not modified.


In [1]:
# ============================================================
# 1. Configuration — PRS Integration and Feature Assembly
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# ------------------------------------------------------------
# 1.1 Sample information
# ------------------------------------------------------------

# Support the current directory and the proposed renamed directory
DATA_DIR_CANDIDATES = [
    PROJECT_ROOT / "Demo_data",
    PROJECT_ROOT / "1.Demo_data",
]

DATA_DIR = next(
    (
        p for p in DATA_DIR_CANDIDATES
        if (p / "1-1.Sample_information/sample_information.csv").is_file()
    ),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate sample_information.csv under Demo_data "
        "or 1.Demo_data."
    )

SAMPLE_INFO_FILE = (
    DATA_DIR
    / "1-1.Sample_information/sample_information.csv"
)


# ------------------------------------------------------------
# 1.2 PRS-CSx
# ------------------------------------------------------------

PRS_FILE = (
    PROJECT_ROOT
    / "Outputs/Step1_PRS/PRS_scoring/"
      "PRS_100_samples_8scores_SUM.csv"
)

PRS_COLUMNS = [
    "PRSCSx_EAS",
    "PRSCSx_EUR",
    "PRSCSx_SAS",
]

# Fixed 1:1:1 combination
PRS_WEIGHTS = {
    "PRSCSx_EAS": 1.0,
    "PRSCSx_EUR": 1.0,
    "PRSCSx_SAS": 1.0,
}

PRS_OUTPUT_COLUMN = "PRS_Score"


# ------------------------------------------------------------
# 1.3 ABD-GNN features
# ------------------------------------------------------------

GNN_FILE = (
    PROJECT_ROOT
    / "Outputs/Step3_GNN/ABD/"
      "ABD_GNN_ActivationVectors.csv"
)


# ------------------------------------------------------------
# 1.4 BMD DenseNet features
# ------------------------------------------------------------

BMD_FILES = {
    "BMD_left_hip": (
        PROJECT_ROOT
        / "Outputs/Step2_DenseNet/BMD_left_hip/"
          "BMD_left_hip_ActivationVectors.csv"
    ),
    "BMD_right_hip": (
        PROJECT_ROOT
        / "Outputs/Step2_DenseNet/BMD_right_hip/"
          "BMD_right_hip_ActivationVectors.csv"
    ),
    "BMD_spine": (
        PROJECT_ROOT
        / "Outputs/Step2_DenseNet/BMD_spine/"
          "BMD_spine_ActivationVectors.csv"
    ),
}


# ------------------------------------------------------------
# 1.5 Clinical predictors
# ------------------------------------------------------------

CLINICAL_COLUMNS = [
    "AGE",
    "SEX",
    "Familyall_numeric",
]


# ------------------------------------------------------------
# 1.6 Expected dataset
# ------------------------------------------------------------

EXPECTED_PARTICIPANTS = 100
FEATURE_DIM = 1024

EXPECTED_SPLITS = {
    "training1": 32,
    "training2": 32,
    "validation": 16,
    "testing": 20,
}


# ------------------------------------------------------------
# 1.7 Output
# ------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT / "Outputs/Step4_Feature_Assembly"
)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "Multimodal_100_samples_PRSCSx_EqualWeight.csv"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1.8 Verify all required input files
# ------------------------------------------------------------

INPUT_FILES = {
    "Sample information": SAMPLE_INFO_FILE,
    "PRS-CSx": PRS_FILE,
    "ABD-GNN": GNN_FILE,
    **BMD_FILES,
}

print("=" * 65)
print("FEATURE ASSEMBLY — INPUT VERIFICATION")
print("=" * 65)

for name, path in INPUT_FILES.items():
    print(
        f"{name:<22}",
        "FOUND" if path.is_file() else "MISSING",
        "|",
        path.relative_to(PROJECT_ROOT)
    )

missing_files = [
    str(path)
    for path in INPUT_FILES.values()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(missing_files)
    )

print("\nAll required input files found.")

FEATURE ASSEMBLY — INPUT VERIFICATION
Sample information     FOUND | Demo_data/1-1.Sample_information/sample_information.csv
PRS-CSx                FOUND | Outputs/Step1_PRS/PRS_scoring/PRS_100_samples_8scores_SUM.csv
ABD-GNN                FOUND | Outputs/Step3_GNN/ABD/ABD_GNN_ActivationVectors.csv
BMD_left_hip           FOUND | Outputs/Step2_DenseNet/BMD_left_hip/BMD_left_hip_ActivationVectors.csv
BMD_right_hip          FOUND | Outputs/Step2_DenseNet/BMD_right_hip/BMD_right_hip_ActivationVectors.csv
BMD_spine              FOUND | Outputs/Step2_DenseNet/BMD_spine/BMD_spine_ActivationVectors.csv

All required input files found.


In [4]:
# ============================================================
# 2. PRS-CSx Integration — Fixed 1:1:1 + Min-Max Scaling
# ============================================================

# ------------------------------------------------------------
# 2.1 Load participant information
# ------------------------------------------------------------

sample_info = pd.read_csv(
    SAMPLE_INFO_FILE,
    dtype={"Demo_ID": str}
)

required_info = [
    "Demo_ID",
    "Dataset",
    "T2D",
    *CLINICAL_COLUMNS,
]

missing = set(required_info) - set(sample_info.columns)
assert not missing, f"Missing sample columns: {missing}"

assert len(sample_info) == EXPECTED_PARTICIPANTS
assert sample_info["Demo_ID"].is_unique

sample_info["Dataset"] = (
    sample_info["Dataset"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert (
    sample_info["Dataset"].value_counts().to_dict()
    == EXPECTED_SPLITS
)

sample_info["T2D"] = pd.to_numeric(
    sample_info["T2D"],
    errors="raise"
).astype(int)

assert set(sample_info["T2D"].unique()) <= {0, 1}

for col in CLINICAL_COLUMNS:
    sample_info[col] = pd.to_numeric(
        sample_info[col],
        errors="raise"
    )

assert sample_info[CLINICAL_COLUMNS].notna().all().all()


# ------------------------------------------------------------
# 2.2 Load ancestry-specific PRS-CSx
# ------------------------------------------------------------

prs_df = pd.read_csv(
    PRS_FILE,
    dtype={"Demo_ID": str}
)

required_prs = ["Demo_ID", *PRS_COLUMNS]

missing = set(required_prs) - set(prs_df.columns)
assert not missing, f"Missing PRS columns: {missing}"

assert prs_df["Demo_ID"].is_unique
assert len(prs_df) == EXPECTED_PARTICIPANTS

assert set(prs_df["Demo_ID"]) == set(sample_info["Demo_ID"])

for col in PRS_COLUMNS:
    prs_df[col] = pd.to_numeric(
        prs_df[col],
        errors="raise"
    )

assert np.isfinite(
    prs_df[PRS_COLUMNS].to_numpy(dtype=float)
).all()


# ------------------------------------------------------------
# 2.3 Fixed equal-weight combination
# ------------------------------------------------------------

# No fitting; no T2D labels used.
# Raw PRS = EAS + EUR + SAS

prs_df["PRS_Score_raw"] = sum(
    PRS_WEIGHTS[col] * prs_df[col]
    for col in PRS_COLUMNS
)

assert np.isfinite(
    prs_df["PRS_Score_raw"].to_numpy(dtype=float)
).all()


# ------------------------------------------------------------
# 2.4 Min-Max Scaling (0–1)
# ------------------------------------------------------------

# IMPORTANT:
# Fit scaling parameters using Training1 + Training2 only.
# Validation and Testing are excluded from fitting.

SCALING_FIT_SPLITS = [
    "training1",
    "training2",
]

training_ids = set(
    sample_info.loc[
        sample_info["Dataset"].isin(SCALING_FIT_SPLITS),
        "Demo_ID"
    ]
)

assert len(training_ids) == 64

train_mask = prs_df["Demo_ID"].isin(training_ids)

assert train_mask.sum() == 64

# Estimate Min/Max from training participants only
prs_min = prs_df.loc[
    train_mask, "PRS_Score_raw"
].min()

prs_max = prs_df.loc[
    train_mask, "PRS_Score_raw"
].max()

prs_range = prs_max - prs_min

if not np.isfinite(prs_range) or prs_range <= 0:
    raise ValueError(
        "Cannot perform Min-Max scaling: "
        "training PRS scores have no valid range."
    )

# Apply the same transformation to all participants
prs_scaled = (
    (prs_df["PRS_Score_raw"] - prs_min)
    / prs_range
)

# Identify observations outside the training range
outside_range = (
    (prs_scaled < 0) | (prs_scaled > 1)
)

n_clipped = int(outside_range.sum())

# Clip to guarantee the final PRS lies within [0, 1]
prs_df[PRS_OUTPUT_COLUMN] = prs_scaled.clip(
    lower=0.0,
    upper=1.0
)

assert np.isfinite(
    prs_df[PRS_OUTPUT_COLUMN].to_numpy(dtype=float)
).all()

assert prs_df[PRS_OUTPUT_COLUMN].between(0, 1).all()


# ------------------------------------------------------------
# 2.5 Merge PRS with sample information
# ------------------------------------------------------------

# Only keep the scaled score for downstream modeling.
# PRS_Score_raw remains in prs_df for inspection.

df_base = sample_info[required_info].merge(
    prs_df[["Demo_ID", PRS_OUTPUT_COLUMN]],
    on="Demo_ID",
    how="left",
    validate="one_to_one"
)

assert len(df_base) == EXPECTED_PARTICIPANTS
assert df_base[PRS_OUTPUT_COLUMN].notna().all()


# ------------------------------------------------------------
# 2.6 Preview
# ------------------------------------------------------------

print("=" * 65)
print("PRS-CSx INTEGRATION")
print("=" * 65)

print("Combination method: Fixed 1:1:1 sum")
print("Ancestry scores:", PRS_COLUMNS)
print("Participants:", len(df_base))

print("\nScaling method: Min-Max (0–1)")
print("Scaling fitted on:", SCALING_FIT_SPLITS)
print("Training PRS minimum:", prs_min)
print("Training PRS maximum:", prs_max)
print("Participants clipped:", n_clipped)

print("\nFinal output PRS column:", PRS_OUTPUT_COLUMN)

print("\nPRS score preview:")

display(
    prs_df[
        [
            "Demo_ID",
            *PRS_COLUMNS,
            "PRS_Score_raw",
            PRS_OUTPUT_COLUMN,
        ]
    ].head(10)
)

print("\nPRS summary by dataset:")

prs_summary = df_base.groupby("Dataset")[
    PRS_OUTPUT_COLUMN
].agg(["count", "min", "mean", "max", "std"])

display(
    prs_summary.reindex(EXPECTED_SPLITS.keys())
)

print("\nParticipant-level base DataFrame:")
display(df_base.head())

print("\nPRS integration and scaling completed.")

PRS-CSx INTEGRATION
Combination method: Fixed 1:1:1 sum
Ancestry scores: ['PRSCSx_EAS', 'PRSCSx_EUR', 'PRSCSx_SAS']
Participants: 100

Scaling method: Min-Max (0–1)
Scaling fitted on: ['training1', 'training2']
Training PRS minimum: 0.01793741
Training PRS maximum: 0.0900768
Participants clipped: 0

Final output PRS column: PRS_Score

PRS score preview:


,Demo_ID,PRSCSx_EAS,PRSCSx_EUR,PRSCSx_SAS,PRS_Score_raw,PRS_Score
0,D001,0.016251,0.013725,0.021142,0.051117,0.459934
1,D002,0.024662,0.000069,0.014813,0.039544,0.299510
2,D003,0.016753,-0.014030,0.032071,0.034794,0.233668
3,D004,0.021090,0.007756,0.030672,0.059518,0.576392
4,D005,0.018364,-0.001820,0.012305,0.028849,0.151256
5,D006,0.022009,0.011101,0.015056,0.048165,0.419021
6,D007,0.015546,0.010838,0.039530,0.065914,0.665058
7,D008,0.009761,-0.009097,0.020892,0.021556,0.050159
8,D009,0.020677,0.006015,0.041046,0.067739,0.690347
9,D010,0.022420,0.005730,0.032140,0.060290,0.587100



PRS summary by dataset:


,count,min,mean,max,std
Dataset,,,,,
training1,32,0.085604,0.415768,0.812034,0.187215
training2,32,0.000000,0.384510,1.000000,0.238725
validation,16,0.143453,0.428458,0.764691,0.192696
testing,20,0.212059,0.474163,0.780660,0.165663



Participant-level base DataFrame:


,Demo_ID,Dataset,T2D,AGE,SEX,Familyall_numeric,PRS_Score
0,D001,training2,0,65,2,0,0.459934
1,D002,testing,0,75,1,0,0.299510
2,D003,training2,1,54,2,0,0.233668
3,D004,training1,1,61,1,0,0.576392
4,D005,training2,1,73,1,0,0.151256



PRS integration and scaling completed.


In [5]:
# ============================================================
# 3. Assemble ABD-GNN and BMD DenseNet Features
# ============================================================

# ------------------------------------------------------------
# 3.1 Feature naming
# ------------------------------------------------------------

GNN_SOURCE_COLUMNS = [
    f"GNN_{i:04d}"
    for i in range(FEATURE_DIM)
]

GNN_OUTPUT_COLUMNS = [
    f"ABD_GNN_{i:04d}"
    for i in range(FEATURE_DIM)
]

BMD_SOURCE_COLUMNS = [
    f"AV_{i:04d}"
    for i in range(FEATURE_DIM)
]

BMD_OUTPUT_COLUMNS = {
    name: [
        f"{name}_AV_{i:04d}"
        for i in range(FEATURE_DIM)
    ]
    for name in BMD_FILES
}


# ------------------------------------------------------------
# 3.2 Helper: Read and verify participant-level features
# ------------------------------------------------------------

def load_feature_table(
    filepath,
    source_columns,
    output_columns,
    source_name
):

    df = pd.read_csv(
        filepath,
        dtype={"Demo_ID": str}
    )

    required = {"Demo_ID", *source_columns}

    missing = required - set(df.columns)

    assert not missing, (
        f"{source_name}: Missing columns: {missing}"
    )

    assert len(df) == EXPECTED_PARTICIPANTS, (
        f"{source_name}: Expected 100 rows, got {len(df)}"
    )

    assert df["Demo_ID"].is_unique, (
        f"{source_name}: Duplicate participant IDs"
    )

    assert set(df["Demo_ID"]) == set(df_base["Demo_ID"]), (
        f"{source_name}: Participant IDs do not match"
    )

    # Verify original partition and outcome consistency
    reference = df_base.set_index("Demo_ID")

    if "Dataset" in df.columns:

        observed_split = (
            df["Dataset"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        expected_split = df["Demo_ID"].map(
            reference["Dataset"]
        )

        assert (
            observed_split.to_numpy()
            == expected_split.to_numpy()
        ).all(), f"{source_name}: Dataset mismatch"

    if "T2D" in df.columns:

        observed_label = pd.to_numeric(
            df["T2D"],
            errors="raise"
        ).to_numpy()

        expected_label = df["Demo_ID"].map(
            reference["T2D"]
        ).to_numpy()

        assert (
            observed_label == expected_label
        ).all(), f"{source_name}: T2D mismatch"

    values = df[source_columns].apply(
        pd.to_numeric,
        errors="raise"
    )

    assert np.isfinite(
        values.to_numpy(dtype=float)
    ).all(), f"{source_name}: Missing or invalid feature values"

    # Keep only the required activation vectors
    result = pd.concat(
        [
            df[["Demo_ID"]].reset_index(drop=True),
            values.reset_index(drop=True),
        ],
        axis=1
    )

    result = result.rename(
        columns=dict(zip(
            source_columns,
            output_columns
        ))
    )

    return result


# ------------------------------------------------------------
# 3.3 Load ABD-GNN
# ------------------------------------------------------------

gnn_features = load_feature_table(
    filepath=GNN_FILE,
    source_columns=GNN_SOURCE_COLUMNS,
    output_columns=GNN_OUTPUT_COLUMNS,
    source_name="ABD-GNN"
)

df_model = df_base.merge(
    gnn_features,
    on="Demo_ID",
    how="left",
    validate="one_to_one"
)

print(
    "ABD-GNN merged:",
    df_model.shape
)


# ------------------------------------------------------------
# 3.4 Load three BMD modalities
# ------------------------------------------------------------

for modality, filepath in BMD_FILES.items():

    bmd_features = load_feature_table(
        filepath=filepath,
        source_columns=BMD_SOURCE_COLUMNS,
        output_columns=BMD_OUTPUT_COLUMNS[modality],
        source_name=modality
    )

    df_model = df_model.merge(
        bmd_features,
        on="Demo_ID",
        how="left",
        validate="one_to_one"
    )

    print(
        f"{modality} merged:",
        df_model.shape
    )


print("\nAll four image modalities assembled.")

ABD-GNN merged: (100, 1031)
BMD_left_hip merged: (100, 2055)
BMD_right_hip merged: (100, 3079)
BMD_spine merged: (100, 4103)

All four image modalities assembled.


In [6]:
# ============================================================
# 4. Final Multimodal DataFrame
# ============================================================

# ------------------------------------------------------------
# 4.1 Define model predictors
# ------------------------------------------------------------

IMAGE_FEATURE_COLUMNS = (
    GNN_OUTPUT_COLUMNS
    + BMD_OUTPUT_COLUMNS["BMD_left_hip"]
    + BMD_OUTPUT_COLUMNS["BMD_right_hip"]
    + BMD_OUTPUT_COLUMNS["BMD_spine"]
)

PREDICTOR_COLUMNS = (
    IMAGE_FEATURE_COLUMNS
    + CLINICAL_COLUMNS
    + [PRS_OUTPUT_COLUMN]
)

META_COLUMNS = [
    "Demo_ID",
    "Dataset",
    "T2D",
]


# ------------------------------------------------------------
# 4.2 Reorder columns
# ------------------------------------------------------------

df_model = df_model[
    META_COLUMNS + PREDICTOR_COLUMNS
].copy()

df_model = (
    df_model
    .sort_values("Demo_ID")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4.3 Final verification
# ------------------------------------------------------------

assert len(df_model) == 100

assert df_model["Demo_ID"].is_unique

assert len(IMAGE_FEATURE_COLUMNS) == 4096

assert len(PREDICTOR_COLUMNS) == 4100

assert df_model.shape == (100, 4103)

assert (
    df_model["Dataset"].value_counts().to_dict()
    == EXPECTED_SPLITS
)

assert df_model[PREDICTOR_COLUMNS].notna().all().all()

assert np.isfinite(
    df_model[PREDICTOR_COLUMNS].to_numpy(dtype=float)
).all()

# No outcome, participant identifier, or dataset label
# is included among the predictors.
assert not (
    set(META_COLUMNS) & set(PREDICTOR_COLUMNS)
)


# ------------------------------------------------------------
# 4.4 Save final dataset
# ------------------------------------------------------------

df_model.to_csv(
    OUTPUT_FILE,
    index=False
)


# ------------------------------------------------------------
# 4.5 Summary
# ------------------------------------------------------------

print("=" * 65)
print("MULTIMODAL FEATURE ASSEMBLY COMPLETED")
print("=" * 65)

print("Participants:", len(df_model))
print("Total columns:", df_model.shape[1])
print("Model predictors:", len(PREDICTOR_COLUMNS))

print("\nPredictor composition:")
print("ABD-GNN:", len(GNN_OUTPUT_COLUMNS))

for modality in BMD_FILES:
    print(
        modality + ":",
        len(BMD_OUTPUT_COLUMNS[modality])
    )

print("Clinical:", len(CLINICAL_COLUMNS))
print("PRS:", 1)

print("\nDataset distribution:")
print(
    df_model["Dataset"]
    .value_counts()
    .reindex(EXPECTED_SPLITS.keys())
)

print("\nOutput:")
print(OUTPUT_FILE.relative_to(PROJECT_ROOT))


# ------------------------------------------------------------
# 4.6 Preview — metadata, clinical features, PRS
# ------------------------------------------------------------

display(
    df_model[
        [
            "Demo_ID",
            "Dataset",
            "T2D",
            "AGE",
            "SEX",
            "Familyall_numeric",
            "PRS_Score",
            "ABD_GNN_0000",
            "BMD_left_hip_AV_0000",
            "BMD_right_hip_AV_0000",
            "BMD_spine_AV_0000",
        ]
    ].head(10)
)

MULTIMODAL FEATURE ASSEMBLY COMPLETED
Participants: 100
Total columns: 4103
Model predictors: 4100

Predictor composition:
ABD-GNN: 1024
BMD_left_hip: 1024
BMD_right_hip: 1024
BMD_spine: 1024
Clinical: 3
PRS: 1

Dataset distribution:
Dataset
training1     32
training2     32
validation    16
testing       20
Name: count, dtype: int64

Output:
Outputs/Step4_Feature_Assembly/Multimodal_100_samples_PRSCSx_EqualWeight.csv


,Demo_ID,Dataset,T2D,AGE,SEX,Familyall_numeric,PRS_Score,ABD_GNN_0000,BMD_left_hip_AV_0000,BMD_right_hip_AV_0000,BMD_spine_AV_0000
0,D001,training2,0,65,2,0,0.459934,-0.011798,0.114165,0.014543,0.003330
1,D002,testing,0,75,1,0,0.299510,-0.011806,0.106138,0.013452,0.002750
2,D003,training2,1,54,2,0,0.233668,-0.004990,0.068251,0.017576,0.005993
3,D004,training1,1,61,1,0,0.576392,-0.006427,0.097509,0.020965,0.001814
4,D005,training2,1,73,1,0,0.151256,-0.007820,0.006203,0.013798,0.004171
5,D006,training2,1,72,1,0,0.419021,-0.002515,0.072893,0.019643,0.005966
6,D007,testing,0,74,1,1,0.665058,-0.010697,0.069223,0.021758,0.003634
7,D008,training2,1,48,2,0,0.050159,-0.003986,0.081633,0.011606,0.008455
8,D009,testing,1,70,2,1,0.690347,-0.016139,0.132890,0.020738,0.001626
9,D010,training1,0,54,1,0,0.587100,-0.011158,0.155917,0.025344,0.002719
